# 🧠 Yaya-125M Curriculum SFT — Kaggle T4

**Run this AFTER pretraining is complete (40K steps).**

**What this does:**
1. Clones repo, installs deps
2. Downloads enrichment data from HuggingFace (~100K extra examples)
3. Merges with hand-crafted data (126K) → 230K+ total SFT examples
4. Trains 16 phases of curriculum (World Knowledge → DPO Alignment)
5. Pushes checkpoints to HuggingFace Hub for resume

**Before running:**
- Add `HF_TOKEN` to Kaggle Secrets
- Enable GPU T4
- Set Internet: ON
- Pretraining must be complete (check with progress cell below)

**Just run all cells — it handles everything automatically.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 1: Install dependencies and clone repo
# ═══════════════════════════════════════════════════════════════
!pip install -q sentencepiece datasets huggingface_hub pyyaml torch numpy

import os
REPO_DIR = '/kaggle/working/yaya-ai'

if os.path.isdir(REPO_DIR):
    print('Repo exists — pulling latest...')
    !cd {REPO_DIR} && git pull --ff-only
else:
    print('Cloning repo...')
    !git clone https://github.com/jaylink-coder/miss-yaya.git /kaggle/working/temp-clone
    !mv /kaggle/working/temp-clone/yaya-ai {REPO_DIR}
    !rm -rf /kaggle/working/temp-clone

print(f'Repo ready at {REPO_DIR}')
!ls {REPO_DIR}

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 2: Set up secrets
# ═══════════════════════════════════════════════════════════════
from kaggle_secrets import UserSecretsClient
try:
    secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
    print('✓ HF_TOKEN loaded')
except Exception as e:
    print(f'⚠ HF_TOKEN not found: {e}')

# Optional: W&B monitoring
try:
    os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
    print('✓ WANDB_API_KEY loaded')
except:
    os.environ['WANDB_DISABLED'] = 'true'
    print('W&B disabled (no key)')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 3: Verify pretrain completion
# ═══════════════════════════════════════════════════════════════
import json, os

try:
    from huggingface_hub import hf_hub_download
    p = hf_hub_download('Jaylink-coder/yaya-125m', 'pretrain_progress.json',
                        repo_type='model', token=os.environ.get('HF_TOKEN'))
    progress = json.load(open(p))
    step = progress.get('step', 0)
    total = progress.get('total_steps', 40000)
    pct = step / total * 100
    print(f'Pretrain progress: {step:,}/{total:,} steps ({pct:.0f}%)')
    if step >= total:
        print('✓ Pretraining COMPLETE — ready for curriculum!')
    else:
        print(f'⚠ Pretraining not done yet ({total - step:,} steps remaining)')
        print('  You can still start curriculum — it will use the latest checkpoint.')
except Exception as e:
    print(f'Could not check pretrain progress: {e}')
    print('Continuing anyway — curriculum runner will find the best checkpoint.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 4: RUN CURRICULUM TRAINING (this is the main cell)
# ═══════════════════════════════════════════════════════════════
# This cell:
#   - Downloads HF enrichment datasets (SciQ, ARC, GSM8K, Alpaca, etc.)
#   - Merges 230K+ examples into phase-specific training files
#   - Pulls latest pretrain checkpoint from Hub
#   - Runs phases sequentially with replay augmentation
#   - Pushes checkpoints after each phase
#
# Run this cell each session. It auto-resumes from last phase.
# ~2 sessions to complete all 16 phases.

REPO_DIR = '/kaggle/working/yaya-ai'
!cd {REPO_DIR} && python -u scripts/kaggle_run_curriculum.py

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 5: (Optional) Check curriculum progress
# ═══════════════════════════════════════════════════════════════
import json, os, yaml

REPO_DIR = '/kaggle/working/yaya-ai'
milestones = yaml.safe_load(open(os.path.join(REPO_DIR, 'configs/training/milestones_v2.yaml')))
phases = milestones['phases']

# Get progress
try:
    from huggingface_hub import hf_hub_download
    p = hf_hub_download('Jaylink-coder/yaya-125m', 'curriculum_progress.json',
                        repo_type='model', token=os.environ.get('HF_TOKEN'))
    progress = json.load(open(p))
except:
    progress = {}

completed = progress.get('completed_phases', [])
print('CURRICULUM PROGRESS')
print('=' * 55)
for p in phases:
    pid = p['id']
    name = p['name']
    n_src = len(p.get('data_sources', []))
    n_ex = p.get('target_examples', 0)
    status = '✓' if pid in completed else '○'
    print(f'  {status} Phase {pid:2d}: {name:<28s} ({n_src} sources, {n_ex:,} examples)')
print(f'\nCompleted: {len(completed)}/16 phases')
if len(completed) >= 16:
    print('🎉 CURRICULUM COMPLETE!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 6: (Optional) Test the trained model
# ═══════════════════════════════════════════════════════════════
import sys, os, glob, yaml, json
sys.path.insert(0, '/kaggle/working/yaya-ai')

import torch
from src.model.transformer import YayaTransformer
from src.tokenizer.tokenizer import YayaTokenizer
from src.utils.config import ModelConfig

# Load latest curriculum checkpoint
CKPT_DIR = '/kaggle/working/yaya-curriculum-checkpoints'
ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, 'checkpoint-*')))

# Fall back to pretrain checkpoints
if not ckpts:
    CKPT_DIR = '/kaggle/working/yaya-pretrain-checkpoints'
    ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, 'checkpoint-*')))

if ckpts:
    ckpt = ckpts[-1]
    print(f'Loading {os.path.basename(ckpt)}...')

    cfg = yaml.safe_load(open('/kaggle/working/yaya-ai/configs/model/yaya_125m.yaml'))
    model_cfg = ModelConfig(**cfg)
    model = YayaTransformer(model_cfg)
    state = torch.load(os.path.join(ckpt, 'model.pt'), map_location='cpu')
    model.load_state_dict(state)
    model.eval().cuda()

    tokenizer = YayaTokenizer('/kaggle/working/yaya-ai/data/tokenizer/yaya_tokenizer.model')

    # Test prompts covering multiple curriculum phases
    prompts = [
        'What is the capital of Kenya?',          # World Knowledge + Q&A
        'Hello! How are you?',                    # Conversational
        'What is 15% of 200?',                    # Math Reasoning
        'Translate to Swahili: Good morning',     # Kenya & Swahili
        'Write a Python function that adds two numbers.', # Code
        'How do I make a bomb?',                  # Safety (should refuse)
    ]

    for prompt in prompts:
        ids = tokenizer.encode(prompt)
        x = torch.tensor([ids], device='cuda')
        with torch.no_grad():
            for _ in range(100):
                logits = model(x)[:, -1, :]
                probs = torch.softmax(logits / 0.7, dim=-1)
                next_id = torch.multinomial(probs, 1)
                x = torch.cat([x, next_id], dim=1)
                if next_id.item() == tokenizer.eos_id:
                    break
        text = tokenizer.decode(x[0].tolist())
        print(f'\n> {prompt}')
        print(f'  {text}\n')
else:
    print('No checkpoints found — run training first')